In [2]:
%load_ext autoreload
%autoreload 2

import numpy as np


import optuna
from sklearn.model_selection import GroupKFold, cross_val_score

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
from TrailPredictor.dataset import Dataset

dataset = Dataset.load(min_distance=40, max_distance=100, segment_width=500)
plotter = dataset.plot

In [ ]:
def build_bpm_zones(
    min_bpm: int, max_bpm: int, n_zones: int
) -> dict[str, tuple[int, int]]:
    """Build heart rate zones based on min and max bpm and number of zones."""
    zone_width = (max_bpm - min_bpm) / n_zones
    zones = {}
    for i in range(n_zones):
        zone_name = f"Z{i+1}"
        zone_min = min_bpm + i * zone_width
        zone_max = min_bpm + (i + 1) * zone_width
        zones[zone_name] = (int(zone_min), int(zone_max))
    zones["Unknown"] = (0, 0)
    return zones

In [ ]:
dataset.data = build_feature(dataset.data, heart_rate_zones)
dataset.target = "pace"
RANDOM_STATE = 42
y = dataset.target
X = dataset.predictors
groups = dataset.data["activity_id"]

In [ ]:
from sklearn.linear_model import LassoCV, LinearRegression
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import Pipeline


# create pipeline => preprocess + lasso + linear regression (on the features selected by lasso)
preprocess = dataset.get_preprocess()
cv = list(GroupKFold(n_splits=5).split(X, y, groups))
pipeline = Pipeline(
    [
        ("preprocess", preprocess),
        (
            "selection",
            SelectFromModel(LassoCV(cv=cv, random_state=RANDOM_STATE)),
        ),
        ("regression", LinearRegression()),
    ]
)
pipeline.fit(X, y)

In [ ]:
HR_LO, HR_HI = 0, 210


def cuts_from_trial(trial, n_zones):
    """Sample ordered heart-rate zone cut points for an optuna trial.

    Parameters
    ----------
    trial : optuna.Trial
        Trial suggesting the relative width of each zone.
    n_zones : int
        Number of zones.

    Returns
    -------
    numpy.ndarray of shape (n_zones - 1,)
        Increasing cut points between ``HR_LO`` and ``HR_HI``, in bpm.
    """
    w = np.array([trial.suggest_float(f"w{i}", 0.05, 1.0) for i in range(n_zones)])
    frac = np.cumsum(w) / w.sum()
    return HR_LO + (HR_HI - HR_LO) * frac[:-1]  # n_zones-1 seuils ordonnés


def make_objective(df, y, groups, model, n_zones, build_features):
    """Build the optuna objective scoring a set of heart-rate zones.

    Parameters
    ----------
    df : pandas.DataFrame
        Segments the features are built from.
    y : array-like
        Target pace of each segment.
    groups : array-like
        Activity of each segment, used for the grouped cross-validation.
    model : estimator
        Scikit-learn model to cross-validate.
    n_zones : int
        Number of zones.
    build_features : callable
        Builds the features from `df` and the zone cut points.

    Returns
    -------
    callable
        Objective returning the cross-validated RMSE of a trial.
    """

    def objective(trial):
        cuts = cuts_from_trial(trial, n_zones)
        X = build_features(
            df, cuts
        )  # ta fonction : zones -> features présent/back/front
        s = cross_val_score(
            model,
            X,
            y,
            groups=groups,
            cv=GroupKFold(5),
            scoring="neg_root_mean_squared_error",
        )
        trial.set_user_attr("cuts", cuts.tolist())
        trial.set_user_attr("rmse_std", s.std())
        return -s.mean()

    return objective


results = {}
for n in range(3, 7):
    study = optuna.create_study(
        direction="minimize", sampler=optuna.samplers.TPESampler(seed=0)
    )
    study.optimize(
        make_objective(df, y, groups, model, n, build_features), n_trials=150
    )
    results[n] = study.best_trial